## Bronze Layer - Unity Catalog Paths

In [0]:
%run ./00_config

###  Kiểm tra file đã upload chưa?

In [0]:
files = dbutils.fs.ls(LANDING_VOLUME)
for f in files:
    size_mb = f.size / 1024 / 1024
    print(f"📄 {f.name} — {size_mb:.1f} MB")

### Tạo Bronze table với schema:

In [0]:
%sql
CREATE TABLE IF NOT EXISTS nyc_taxi_project.bronze.yellow_taxi
(
  VendorID              LONG,
  tpep_pickup_datetime  TIMESTAMP,
  tpep_dropoff_datetime TIMESTAMP,
  passenger_count       DOUBLE,
  trip_distance         DOUBLE,
  RatecodeID            DOUBLE,
  store_and_fwd_flag    STRING,
  PULocationID          LONG,
  DOLocationID          LONG,
  payment_type          LONG,
  fare_amount           DOUBLE,
  extra                 DOUBLE,
  mta_tax               DOUBLE,
  tip_amount            DOUBLE,
  tolls_amount          DOUBLE,
  improvement_surcharge DOUBLE,
  total_amount          DOUBLE,
  congestion_surcharge  DOUBLE,
  Airport_fee           DOUBLE,
  -- Metadata columns
  ingested_at           TIMESTAMP,
  source_file           STRING,
  ingest_date           DATE
)
USING DELTA
COMMENT 'Bronze layer - raw NYC Yellow Taxi data'
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true');

### Auto Loader Streaming

In [0]:
from pyspark.sql import functions as F

def process_bronze():

    query = (spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", "parquet")
                .option("cloudFiles.schemaLocation", BRONZE_CHECKPOINT + "schema")
                .option("cloudFiles.inferColumnTypes", "true")
                .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
                .option("ignoreMissingFiles", "true")
                .load(LANDING_VOLUME)
                .withColumn("ingested_at", F.current_timestamp())
                .withColumn("source_file", F.col("_metadata.file_path"))
                .withColumn("ingest_date", F.to_date(F.current_timestamp()))
             .writeStream
                .option("checkpointLocation", BRONZE_CHECKPOINT)
                .option("mergeSchema", "true")
                .trigger(availableNow=True)
                .table(BRONZE_TABLE))

    query.awaitTermination()

process_bronze()